# Cache Paths

> Per-(input-content, config) deterministic cache directories for capability outputs

Phase 1 Q3 Layer B (Track 19 companion). Capabilities writing cacheable outputs
(ffmpeg segments, demucs stems, lavasr enhanced audio, transcription jobs)
historically derived output paths from `(capability_data_dir, input_stem)` alone
— different configs silently overwrote each other and stale-artifacts
accumulated when a new run produced fewer outputs than the previous run
(the user's reported ffmpeg segmentation bug).

`cache_dir_for_config(capability_data_dir, input_path, action, config_dict)`
returns a deterministic per-(input-content, config) directory:

```
<capability_data_dir>/<action>/<sanitized-stem>/<input_hash[:6]>_<config_hash[:12]>/
```

Three properties matter:

1. **Content-keyed input identity.** The input's content hash (with stat-cache
   memoization on `(path, mtime, size)`) goes into the key. Two files with the
   same stem in different directories produce different keys. A modify-in-place
   on the same path produces a different key.

2. **Config-keyed by `compute_config_hash`.** Different configs go to different
   directories — no silent overwrite, no stale-artifact accumulation.

3. **Sequence-friendly.** When capability A's output feeds capability B (the
   `submit_sequence` shape), A's output path embeds A's config_hash. When A's
   config changes, A's output content changes, B's content-hash of A's output
   changes, so B's cache key changes too — chained invalidation happens
   automatically without explicit lineage tracking.

Sibling APIs: `list_cache_entries` for inspection, `prune_cache_for_input` for
operator-controlled cleanup of stale config variants.

In [ ]:
#| default_exp utils.cache_paths

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import logging
import os
import re
import shutil
import sqlite3
import threading
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Set, Union

from cjm_plugin_system.core.empirical_store import compute_config_hash
from cjm_plugin_system.utils.hashing import hash_file

_logger = logging.getLogger(__name__)

## _sanitize_stem

Replace filesystem-unsafe characters in an input file's stem so the
resulting directory name is portable across Linux / macOS / Windows.
Length-cap mitigates pathologically long source filenames.

In [ ]:
#| export
# Filesystem-portable max stem length. Linux + macOS handle 255-byte filenames;
# Windows MAX_PATH considerations + Unicode encoding make 100 chars a safe
# operator-readable cap. Operators ARE going to inspect these directories;
# long stems are noise.
_MAX_STEM_LEN = 100

# Replace characters that are problematic on at least one major filesystem.
# Windows reserves these explicitly; Linux/macOS tolerate them but operators
# don't want them in path components either.
_UNSAFE_CHARS = re.compile(r'[<>:"/\\|?*\x00-\x1f]')


def _sanitize_stem(
    input_path: Union[str, Path],  # Path whose stem to sanitize
) -> str:                            # Filesystem-portable stem string
    """Return a filesystem-portable, length-capped version of input_path's stem.

    Steps: take `Path(input_path).stem`, replace unsafe characters with `_`,
    strip leading/trailing dots + whitespace (Windows path-component rule),
    length-cap to `_MAX_STEM_LEN`. Empty stems (degenerate paths like `.`)
    return `_` so callers always get a usable path component.
    """
    raw = Path(str(input_path)).stem
    if not raw:
        return "_"
    cleaned = _UNSAFE_CHARS.sub("_", raw)
    cleaned = cleaned.strip(". \t")
    if not cleaned:
        return "_"
    return cleaned[:_MAX_STEM_LEN]

## Stat-cache for content hashes

Computing a SHA-256 over a multi-hour podcast WAV (~1–2 GB) takes a few
seconds even with `hash_file`'s streaming reader. Per-cache-lookup hashing
would be untenable for chained-capability workflows where the same input might
be referenced dozens of times in a single workflow run.

The substrate maintains a small SQLite stat-cache at
`<substrate_data_dir>/input_hash_cache.db` mapping
`(absolute_path, mtime_ns, size)` → `content_hash`. Lookups that hit the
cache return in microseconds; cold lookups compute the hash once and write
it back.

The cache uses a module-level `threading.Lock` to serialize SQLite writes
(SQLite handles concurrent reads fine, but writes from multiple threads
need coordination at the Python layer to avoid `database is locked`
errors). Reads are still fast because SQLite's WAL mode (set on connect)
allows concurrent readers + one writer.

`mtime_ns` is preferred over `mtime` (float seconds) — nanosecond precision
distinguishes a fast write-twice operation that a 1-second mtime resolution
would conflate. `size` is the secondary check — paranoid defense against
filesystem mtime resolution issues.

In [ ]:
#| export
_stat_cache_lock = threading.Lock()

In [ ]:
#| export
def _get_stat_cache_path() -> Optional[Path]:
    """Locate the stat-cache SQLite path under the substrate's data dir.

    Returns the configured `cfg.data_dir / "input_hash_cache.db"` when the
    substrate config is available, falling back to `~/.cjm/input_hash_cache.db`
    when get_config() raises (e.g., during early-init tests). Returns None
    only when neither resolves — callers then skip caching entirely.
    """
    try:
        from cjm_plugin_system.core.config import get_config
        cfg = get_config()
        if cfg.data_dir is not None:
            return cfg.data_dir / "input_hash_cache.db"
    except Exception:
        pass
    # Fallback: per-user default. Match the convention CR-2's
    # LocalCapabilityConfigStore + CR-7's LocalEmpiricalResourceStore use.
    home = Path.home()
    return home / ".cjm" / "input_hash_cache.db"

In [ ]:
#| export
def _ensure_stat_cache_schema(conn: sqlite3.Connection) -> None:
    """Create the stat-cache table + indices if not present."""
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS input_hash_cache (
            path TEXT NOT NULL,
            mtime_ns INTEGER NOT NULL,
            size INTEGER NOT NULL,
            content_hash TEXT NOT NULL,
            PRIMARY KEY (path, mtime_ns, size)
        )
    """)
    conn.commit()

In [ ]:
#| export
def _hash_input_with_stat_cache(
    input_path: Path,                      # File whose content hash we need
    cache_path: Optional[Path] = None,     # Explicit cache DB path (default: substrate-resolved)
    *,
    skip_cache: bool = False,              # If True, bypass cache entirely (compute always)
) -> str:                                   # Hash string in "algo:hexdigest" format
    """Return the SHA-256 content hash of `input_path`, with stat-cache memoization.

    Fast path: stat the file, look up `(absolute_path, mtime_ns, size)` in the
    SQLite cache; cache hit returns in microseconds. Cold path: `hash_file`
    streams the file content, then writes the result to the cache.

    `skip_cache=True` bypasses the cache entirely — useful for callers that
    KNOW the file content changed (e.g., a capability just wrote it and wants
    to record the canonical hash without polluting the cache with intermediate
    states).

    File not found / unreadable → propagates the underlying OSError. Cache
    DB errors are LOG-and-FALLBACK to direct hashing — the cache is an
    optimization, not a correctness invariant.
    """
    p = Path(input_path).resolve()
    if not p.exists():
        # Let hash_file's open() raise the FileNotFoundError with a clear path.
        return hash_file(p)

    if skip_cache:
        return hash_file(p)

    try:
        st = p.stat()
        mtime_ns = st.st_mtime_ns
        size = st.st_size
    except OSError as e:
        _logger.debug(f"stat failed for {p!r}: {e}; falling back to direct hash")
        return hash_file(p)

    db_path = cache_path or _get_stat_cache_path()
    if db_path is None:
        return hash_file(p)
    # Ensure the parent dir exists; the substrate creates other store dirs
    # the same way in CR-2 / CR-7.
    try:
        db_path.parent.mkdir(parents=True, exist_ok=True)
    except OSError as e:
        _logger.debug(f"cache dir {db_path.parent!r} unavailable: {e}")
        return hash_file(p)

    with _stat_cache_lock:
        try:
            conn = sqlite3.connect(str(db_path))
            try:
                _ensure_stat_cache_schema(conn)
                row = conn.execute(
                    "SELECT content_hash FROM input_hash_cache "
                    "WHERE path = ? AND mtime_ns = ? AND size = ?",
                    (str(p), mtime_ns, size),
                ).fetchone()
                if row is not None:
                    return row[0]
                # Cold path: compute + cache.
                content_hash = hash_file(p)
                conn.execute(
                    "INSERT OR REPLACE INTO input_hash_cache "
                    "(path, mtime_ns, size, content_hash) VALUES (?, ?, ?, ?)",
                    (str(p), mtime_ns, size, content_hash),
                )
                conn.commit()
                return content_hash
            finally:
                conn.close()
        except sqlite3.Error as e:
            _logger.warning(
                f"stat-cache DB error at {db_path!r}: {e}; falling back to direct hash"
            )
            return hash_file(p)

## cache_dir_for_config

The main entry point. Returns (and optionally creates) a deterministic
per-(input-content, config) cache directory.

Capabilities use this in lieu of hand-rolled `<capability_data_dir>/<action>/<stem>`
output-path derivation. The user's ffmpeg segmentation bug (`segment_audio`
with different `max_segment_duration` values overwriting each other in the
same directory) is the canonical motivating example — fixing it requires
the config to enter the cache key, which this helper makes mandatory by
construction.

In [ ]:
#| export
def cache_dir_for_config(
    capability_data_dir: Union[str, Path],     # The capability's own data subdirectory (typically <cfg.capability_data_dir>/<capability_name>)
    input_path: Union[str, Path],           # The input file the capability operates on
    action: str,                            # The capability action name (e.g., "segment_audio", "convert", "execute")
    config_dict: Dict[str, Any],            # The capability's effective config for this action
    *,
    input_hash_length: int = 6,             # Truncation length for the input content hash in the directory name
    config_hash_length: int = 12,           # Truncation length for the config hash in the directory name
    create: bool = True,                    # Auto-create the directory (parents=True, exist_ok=True)
    hash_input_content: bool = True,        # If False, hash str(input_path) instead (e.g., URL inputs)
    skip_input_cache: bool = False,         # If True, bypass the stat-cache (always recompute content hash)
) -> Path:                                   # The deterministic cache directory path
    """Return (and optionally create) a per-(input-content, config) cache directory.

    Path layout::

        <capability_data_dir>/<action>/<sanitized-stem>/<input_hash[:N]>_<config_hash[:M]>/

    The same `(input_content, action, config_dict)` always resolves to the same
    path; any change to input content OR config produces a different path. This
    means:

    1. Different configs go to different directories — no silent overwrite.
    2. Stale-artifact accumulation is impossible — each unique
       `(input_content, config)` tuple has its OWN directory.
    3. For chained capability sequences, upstream config changes propagate through
       content changes: if capability A's output content depends on A's config and
       capability B reads that output, B's cache key automatically reflects A's
       config indirectly.

    `hash_input_content=False` switches to hashing the string form of
    `input_path` instead of file content — for capabilities whose "input" is a URL,
    a database row ID, or another non-file identifier. Sequence chaining via
    content propagation only works for true file inputs.

    `skip_input_cache=True` recomputes the input content hash even if the
    stat-cache has a record. Useful for capabilities that just wrote the input file
    and want to record its canonical hash without stale-cache risk.

    Raises FileNotFoundError if `input_path` doesn't exist and
    `hash_input_content=True`. Raises OSError on directory-create failure
    when `create=True`.
    """
    base = Path(capability_data_dir)
    stem = _sanitize_stem(input_path)
    if hash_input_content:
        input_hash = _hash_input_with_stat_cache(
            Path(input_path), skip_cache=skip_input_cache,
        )
    else:
        # Non-file input — hash the string form. Stem still derived from the
        # input string so operator-inspectable directories remain readable.
        from cjm_plugin_system.utils.hashing import hash_bytes
        input_hash = hash_bytes(str(input_path).encode("utf-8"))
    # Both hashes come back as "sha256:hexdigest"; strip the algo prefix
    # before truncation so the directory component is pure hex.
    input_hex = input_hash.split(":", 1)[1] if ":" in input_hash else input_hash
    config_hash = compute_config_hash(config_dict)
    config_hex = config_hash.split(":", 1)[1] if ":" in config_hash else config_hash
    leaf = f"{input_hex[:input_hash_length]}_{config_hex[:config_hash_length]}"
    out_dir = base / action / stem / leaf
    if create:
        out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

## list_cache_entries + prune_cache_for_input

Operator-facing affordances for inspecting and cleaning up the cache. The
`<capability_data_dir>/<action>/<stem>/` parent contains one directory per
unique config variant the capability has been run with. `list_cache_entries`
enumerates them; `prune_cache_for_input` deletes them (optionally
preserving a specified set).

In [ ]:
#| export
def list_cache_entries(
    capability_data_dir: Union[str, Path],  # The capability's own data subdirectory
    input_path: Union[str, Path],        # The input file whose cache entries to list
    action: str,                          # The capability action name
) -> List[Path]:                          # All config-hash directories for this (input, action)
    """Enumerate all per-config cache directories for a given (input, action).

    Returns the paths of every `<input_hash>_<config_hash>` directory under
    `<capability_data_dir>/<action>/<sanitized-stem>/`. Each entry corresponds to
    a unique `(input_content, config)` tuple — operators can inspect their
    contents, diff them, or pass selected ones to `prune_cache_for_input` to
    keep them through a sweep.

    Returns an empty list if the parent directory doesn't exist (capability never
    ran this action for this input).
    """
    base = Path(capability_data_dir)
    stem = _sanitize_stem(input_path)
    parent = base / action / stem
    if not parent.exists():
        return []
    return sorted(p for p in parent.iterdir() if p.is_dir())


def prune_cache_for_input(
    capability_data_dir: Union[str, Path],  # The capability's own data subdirectory
    input_path: Union[str, Path],        # The input file whose cache entries to prune
    action: str,                          # The capability action name
    *,
    keep: Optional[Set[Path]] = None,    # Paths to preserve through the sweep (returns by list_cache_entries)
    dry_run: bool = False,                # If True, return what WOULD be deleted without touching filesystem
) -> List[Path]:                          # Paths that were (or would be) deleted
    """Delete per-config cache directories for `(input, action)`, optionally
    preserving a `keep` set.

    Pairs with `list_cache_entries` for inspect-then-prune workflows: list
    candidates, choose which to keep, then call prune with the keep set.
    `keep=None` deletes ALL entries.

    `dry_run=True` returns the would-delete list without touching the
    filesystem — useful for operator confirmation before destructive ops.

    Returns the list of deleted (or would-delete) paths.
    """
    keep_set = keep or set()
    targets = list_cache_entries(capability_data_dir, input_path, action)
    deleted: List[Path] = []
    for entry in targets:
        if entry in keep_set:
            continue
        if not dry_run:
            shutil.rmtree(entry, ignore_errors=True)
        deleted.append(entry)
    return deleted

## Tests

Exercise the helpers end-to-end against a tempdir + tempfile so the
cache_paths module doesn't depend on any specific capability's data dir.

In [ ]:
#| hide
import tempfile
import time as _t


def _q3_cache_dir_basic_check():
    """Same (input, action, config) → same dir; different config → different dir."""
    with tempfile.TemporaryDirectory() as tmp_root:
        capability_data = Path(tmp_root) / "capability_data"
        # Make an input file with deterministic content
        input_file = Path(tmp_root) / "podcast.mp3"
        input_file.write_bytes(b"fake mp3 content for testing")

        cfg_a = {"max_segment_duration": 300}
        cfg_b = {"max_segment_duration": 600}

        # Same config → same directory (idempotent)
        dir1 = cache_dir_for_config(capability_data, input_file, "segment_audio", cfg_a)
        dir2 = cache_dir_for_config(capability_data, input_file, "segment_audio", cfg_a)
        assert dir1 == dir2, "same config → same dir"
        assert dir1.exists(), "create=True default must mkdir"
        assert dir1.parent.name == "podcast", "stem in path: " + str(dir1)
        assert dir1.parent.parent.name == "segment_audio", "action in path"

        # Different config → different directory (the ffmpeg segment bug is fixed)
        dir3 = cache_dir_for_config(capability_data, input_file, "segment_audio", cfg_b)
        assert dir3 != dir1, "different config → different dir"
        assert dir3.parent == dir1.parent, "same stem parent"

        # Different action → different parent path
        dir4 = cache_dir_for_config(capability_data, input_file, "convert", cfg_a)
        assert dir4.parent.parent != dir1.parent.parent, "different action → different parent"


def _q3_cache_dir_same_stem_collision_check():
    """Two different files with the same stem produce DIFFERENT cache keys
    because the input content hash distinguishes them (the user's
    `short_test_audio.mp3` cross-directory concern)."""
    with tempfile.TemporaryDirectory() as tmp_root:
        capability_data = Path(tmp_root) / "capability_data"
        dir_a = Path(tmp_root) / "podcasts"
        dir_b = Path(tmp_root) / "lectures"
        dir_a.mkdir()
        dir_b.mkdir()

        # Same stem, different content
        file_a = dir_a / "short_test_audio.mp3"
        file_b = dir_b / "short_test_audio.mp3"
        file_a.write_bytes(b"podcast content")
        file_b.write_bytes(b"lecture content")

        cfg = {"sample_rate": 16000}
        dir_for_a = cache_dir_for_config(capability_data, file_a, "convert", cfg)
        dir_for_b = cache_dir_for_config(capability_data, file_b, "convert", cfg)

        assert dir_for_a != dir_for_b, "same-stem different-content must produce different cache dirs"
        # Stem component is the same (operator-readable)
        assert dir_for_a.parent.name == dir_for_b.parent.name == "short_test_audio"
        # Leaf differs (different input hash component)
        assert dir_for_a.name != dir_for_b.name


def _q3_cache_dir_modify_in_place_check():
    """Modifying a file in place changes its cache key (mtime invalidates stat-cache)."""
    with tempfile.TemporaryDirectory() as tmp_root:
        capability_data = Path(tmp_root) / "capability_data"
        input_file = Path(tmp_root) / "audio.wav"
        input_file.write_bytes(b"original content")

        cfg = {"action": "x"}
        dir_before = cache_dir_for_config(capability_data, input_file, "x", cfg)

        # Modify the file. Sleep briefly to ensure mtime advances on all
        # filesystems (some have second-resolution mtime; nanosecond is the
        # standard but test fixtures should be defensive).
        _t.sleep(0.05)
        input_file.write_bytes(b"NEW content - entirely different")

        dir_after = cache_dir_for_config(capability_data, input_file, "x", cfg)
        assert dir_before != dir_after, "modify-in-place must change cache key"


def _q3_sequence_chaining_check():
    """Capability A's output → capability B's input: when A's config changes, A's
    output content changes, so B's cache key auto-invalidates. Verifies
    the chained-capability sequence UX the user asked about."""
    with tempfile.TemporaryDirectory() as tmp_root:
        ffmpeg_data = Path(tmp_root) / "ffmpeg_data"
        voxtral_data = Path(tmp_root) / "voxtral_data"
        original_audio = Path(tmp_root) / "podcast.mp3"
        original_audio.write_bytes(b"original podcast")

        # ffmpeg.convert(audio, config_a) produces output_a
        ffmpeg_cfg_16k = {"sample_rate": 16000}
        ffmpeg_out_dir_a = cache_dir_for_config(
            ffmpeg_data, original_audio, "convert", ffmpeg_cfg_16k,
        )
        # Simulate ffmpeg writing its output
        ffmpeg_output_a = ffmpeg_out_dir_a / "podcast.wav"
        ffmpeg_output_a.write_bytes(b"converted at 16k")

        # voxtral.execute(output_a, voxtral_cfg) → voxtral cache key for output_a
        voxtral_cfg = {"model": "voxtral-mini-3b"}
        voxtral_out_dir_for_a = cache_dir_for_config(
            voxtral_data, ffmpeg_output_a, "execute", voxtral_cfg,
        )

        # Now: ffmpeg.convert(audio, config_b) produces output_b
        ffmpeg_cfg_24k = {"sample_rate": 24000}
        ffmpeg_out_dir_b = cache_dir_for_config(
            ffmpeg_data, original_audio, "convert", ffmpeg_cfg_24k,
        )
        ffmpeg_output_b = ffmpeg_out_dir_b / "podcast.wav"
        ffmpeg_output_b.write_bytes(b"converted at 24k - different content")

        # voxtral.execute(output_b, voxtral_cfg) — voxtral_cfg UNCHANGED
        voxtral_out_dir_for_b = cache_dir_for_config(
            voxtral_data, ffmpeg_output_b, "execute", voxtral_cfg,
        )

        # voxtral's cache dir DIFFERS because the input content differs,
        # even though voxtral's own config is unchanged. This is the
        # automatic-chain-invalidation property.
        assert voxtral_out_dir_for_a != voxtral_out_dir_for_b, (
            "upstream config change must propagate to downstream cache key"
        )


def _q3_create_false_returns_path_without_mkdir():
    """create=False returns the path without creating the directory."""
    with tempfile.TemporaryDirectory() as tmp_root:
        capability_data = Path(tmp_root) / "capability_data"
        input_file = Path(tmp_root) / "x.wav"
        input_file.write_bytes(b"x")

        out_dir = cache_dir_for_config(
            capability_data, input_file, "act", {"k": 1}, create=False,
        )
        assert not out_dir.exists(), "create=False must not mkdir"


def _q3_hash_input_content_false_uses_path_string():
    """hash_input_content=False hashes the path string (e.g., URL inputs)."""
    with tempfile.TemporaryDirectory() as tmp_root:
        capability_data = Path(tmp_root) / "capability_data"
        # Non-file input: a URL
        url1 = "https://example.com/audio-A.mp3"
        url2 = "https://example.com/audio-B.mp3"

        cfg = {"x": 1}
        dir1 = cache_dir_for_config(
            capability_data, url1, "fetch", cfg, hash_input_content=False,
        )
        dir2 = cache_dir_for_config(
            capability_data, url2, "fetch", cfg, hash_input_content=False,
        )
        assert dir1 != dir2, "different URL strings → different cache dirs"
        # Idempotent for the same URL
        dir1_again = cache_dir_for_config(
            capability_data, url1, "fetch", cfg, hash_input_content=False,
        )
        assert dir1 == dir1_again


def _q3_sanitize_stem_check():
    """_sanitize_stem replaces unsafe characters + handles edge cases."""
    assert _sanitize_stem("normal.mp3") == "normal"
    assert _sanitize_stem("01 - Chapter 1.mp3") == "01 - Chapter 1"
    # Filesystem-unsafe characters get replaced with _
    s = _sanitize_stem("weird<>:|?*.mp3")
    assert "<" not in s and ">" not in s and "|" not in s
    # Length cap
    long_stem = _sanitize_stem("x" * 500 + ".mp3")
    assert len(long_stem) <= _MAX_STEM_LEN
    # Degenerate path returns "_"
    assert _sanitize_stem(".") == "_"
    # Leading/trailing dots and whitespace stripped (Windows convention)
    assert _sanitize_stem("  ..hello..  .mp3") == "hello"


def _q3_list_and_prune_companions():
    """list_cache_entries + prune_cache_for_input work in tandem."""
    with tempfile.TemporaryDirectory() as tmp_root:
        capability_data = Path(tmp_root) / "capability_data"
        input_file = Path(tmp_root) / "audio.wav"
        input_file.write_bytes(b"x")

        # Create THREE cache entries via three different configs
        d1 = cache_dir_for_config(capability_data, input_file, "act", {"k": 1})
        d2 = cache_dir_for_config(capability_data, input_file, "act", {"k": 2})
        d3 = cache_dir_for_config(capability_data, input_file, "act", {"k": 3})
        # Drop a marker file in each so we can verify prune deletion
        (d1 / "marker").write_text("1")
        (d2 / "marker").write_text("2")
        (d3 / "marker").write_text("3")

        # list_cache_entries surfaces all three
        entries = list_cache_entries(capability_data, input_file, "act")
        assert set(entries) == {d1, d2, d3}, "unexpected: " + str(entries)

        # Dry-run prune doesn't touch the filesystem
        would_delete = prune_cache_for_input(
            capability_data, input_file, "act", keep={d2}, dry_run=True,
        )
        assert set(would_delete) == {d1, d3}
        assert d1.exists() and d2.exists() and d3.exists()

        # Real prune deletes d1 and d3, preserves d2
        deleted = prune_cache_for_input(capability_data, input_file, "act", keep={d2})
        assert set(deleted) == {d1, d3}
        assert not d1.exists() and not d3.exists()
        assert d2.exists()

        # list after prune surfaces only d2
        remaining = list_cache_entries(capability_data, input_file, "act")
        assert remaining == [d2]


def _q3_stat_cache_round_trip():
    """_hash_input_with_stat_cache returns the same hash on second call (cache hit)."""
    with tempfile.TemporaryDirectory() as tmp_root:
        cache_db = Path(tmp_root) / "cache.db"
        input_file = Path(tmp_root) / "test.bin"
        input_file.write_bytes(b"deterministic content for hashing")

        # First call: cold (writes to cache)
        h1 = _hash_input_with_stat_cache(input_file, cache_path=cache_db)
        # Cache DB should now exist
        assert cache_db.exists(), "cache DB should have been created"

        # Second call: warm (reads from cache) — same hash
        h2 = _hash_input_with_stat_cache(input_file, cache_path=cache_db)
        assert h1 == h2, "warm cache must return the same hash"

        # Modify the file → mtime changes → cache miss → fresh hash
        _t.sleep(0.05)
        input_file.write_bytes(b"DIFFERENT content")
        h3 = _hash_input_with_stat_cache(input_file, cache_path=cache_db)
        assert h3 != h1, "modify-in-place must produce different hash"

        # And the new hash is itself cached
        h4 = _hash_input_with_stat_cache(input_file, cache_path=cache_db)
        assert h4 == h3, "new hash must be cached too"


def _q3_skip_cache_bypass():
    """skip_cache=True bypasses the cache entirely."""
    with tempfile.TemporaryDirectory() as tmp_root:
        cache_db = Path(tmp_root) / "cache.db"
        input_file = Path(tmp_root) / "test.bin"
        input_file.write_bytes(b"content")

        h1 = _hash_input_with_stat_cache(input_file, cache_path=cache_db)
        h2 = _hash_input_with_stat_cache(input_file, cache_path=cache_db, skip_cache=True)
        # Both compute the same hash, but skip_cache=True bypasses the lookup
        assert h1 == h2


_q3_cache_dir_basic_check()
_q3_cache_dir_same_stem_collision_check()
_q3_cache_dir_modify_in_place_check()
_q3_sequence_chaining_check()
_q3_create_false_returns_path_without_mkdir()
_q3_hash_input_content_false_uses_path_string()
_q3_sanitize_stem_check()
_q3_list_and_prune_companions()
_q3_stat_cache_round_trip()
_q3_skip_cache_bypass()
print("Q3 cache_paths (Layer B): PASS")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()